In [9]:
import kagglehub

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("spotifytracks.csv")

df = df.rename(columns={"artists": "artist_name"})

features_cols = [
    'danceability', 'energy', 'valence',
    'tempo', 'acousticness',
    'speechiness', 'liveness',
    'instrumentalness', 'loudness'
]

# Drop missing values
df = df.dropna(subset=features_cols)

# Remove unpopular songs
df = df[df['popularity'] >= 50]

# Clean names
df['track_name_clean'] = df['track_name'].str.lower().str.strip()
df['artist_name_clean'] = df['artist_name'].str.lower().str.strip()

# Remove exact duplicates
df = df.drop_duplicates(subset=['track_name_clean', 'artist_name_clean'])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features_cols])

query_songs = [
    ("Death with Dignity", "Sufjan Stevens"),
    ("Motion Sickness", "Phoebe Bridgers"),
    ("Lean on Me", "Bill Withers"),
    ("Nude","Radiohead"),
]

def find_song_index(track, artist):
    track = track.lower().strip()
    artist = artist.lower().strip()

    matches = df[
        df['track_name_clean'].str.contains(track, na=False) &
        df['artist_name_clean'].str.contains(artist, na=False)
    ]

    if len(matches) == 0:
        print(f"\nNo match found for '{track}' by '{artist}'")
        return None

    return matches.index[0]

def get_top_similar_songs(idx, top_n=10):
    query_vector = X_scaled[df.index.get_loc(idx)].reshape(1, -1)
    similarities = cosine_similarity(query_vector, X_scaled)[0]

    sim_series = pd.Series(similarities, index=df.index)

    # Query song info
    query_track = df.loc[idx, 'track_name_clean']
    query_artist = df.loc[idx, 'artist_name_clean']

    # Remove same song 
    mask = ~(
        (df['track_name_clean'] == query_track) &
        (df['artist_name_clean'] == query_artist)
    )

    sim_series = sim_series[mask]

    sim_series = sim_series.sort_values(ascending=False)

    seen = set()
    rows = []

    for i in sim_series.index:
        track = df.loc[i, 'track_name']
        artist = df.loc[i, 'artist_name']

        key = (track.lower().strip(), artist.lower().strip())

        if key in seen:
            continue

        seen.add(key)

        rows.append({
            "Track": track,
            "Artist": artist,
            "Similarity": round(sim_series[i], 4)
        })

        if len(rows) >= top_n:
            break

    result_df = pd.DataFrame(rows)
    result_df.insert(0, "Rank", range(1, len(result_df) + 1))

    return result_df

for track, artist in query_songs:
    idx = find_song_index(track, artist)

    print(f"\nTop 10 songs similar to '{track}' by {artist}:\n")

    if idx is None:
        print("Song not found in dataset.\n")
        continue

    table = get_top_similar_songs(idx)

    print(table.to_string(index=False))

    # Save results
    filename = track.replace(" ", "_").replace("+", "").replace("'", "")
    table.to_csv(f"{filename}_similar_songs.csv", index=False)


Top 10 songs similar to 'Death with Dignity' by Sufjan Stevens:

 Rank                              Track                    Artist  Similarity
    1                     The Only Thing            Sufjan Stevens      0.9886
    2                               太陽さん               Ichiko Aoba      0.9880
    3                     Chitthi Bhitra         Sajjan Raj Vaidya      0.9739
    4                Blowin' in the Wind                 Bob Dylan      0.9737
    5                       Song For You             Alexi Murdoch      0.9706
    6 For All You Give (feat. Lucy Rose) The Paper Kites;Lucy Rose      0.9569
    7                Have We Met Before?  Tom Rosenthal;Fenne Lily      0.9541
    8            Jaan Nisaar (LoFi Flip)       Tashif;Milli Mourya      0.9537
    9                          Canteiros                    Fagner      0.9527
   10                               Arms           The Paper Kites      0.9526

Top 10 songs similar to 'Motion Sickness' by Phoebe Bridgers:

 